# WT-Übung 5: 2D-Normalverteilung

Notebook-Fassung von `u5_2D_gaussian.py` (das Original braucht PyQt5 und ein eigenes Fenster).

Gezeigt werden zufällige Realisierungen einer zweidimensionalen Normalverteilung mit den Standardabweichungen $\sigma_1$, $\sigma_2$ und dem Korrelationskoeffizienten $\rho$, dazu die Höhenlinie der Dichte für $K = 9$ (rote Ellipse).

**Play** lässt die Punktwolke wachsen; die Schieberegler ändern die Verteilung.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse
import ipywidgets as widgets

%matplotlib inline

In [ ]:
def contour_ellipse(o1, o2, r, K=3 ** 2):
    """Höhenlinie der Dichte als Ellipse (gleiche Rechnung wie im Original)."""
    sqrt, sin, cos, pi = np.sqrt, np.sin, np.cos, np.pi
    g = 0.5 * np.arctan(2 * r * o1 * o2 / (o2**2 - o1**2)) if o1 != o2 else pi / 4
    a = o1 * o2 * sqrt(K * (1 - r**2) / (
        (o1 * sin(g)) ** 2 + (o2 * cos(g)) ** 2 + 2 * r * o1 * o2 * sin(g) * cos(g)))
    b = o1 * o2 * sqrt(K * (1 - r**2) / (
        (o1 * cos(g)) ** 2 + (o2 * sin(g)) ** 2 - 2 * r * o1 * o2 * sin(g) * cos(g)))
    return Ellipse(xy=(0, 0), width=2 * a, height=2 * b, angle=-180 / pi * g,
                   facecolor='none', edgecolor='r', zorder=2, linewidth=2)


def gaussian_samples(o1, o2, r, n, seed=0):
    """n Punkte mit Korrelation r aus zwei unabhängigen N(0,1)-Vektoren."""
    x, y = np.random.default_rng(seed).normal(0, 1, (2, n))
    r1, r2 = np.sqrt((1 + r) / 2), np.sqrt((1 - r) / 2)
    return o1 * (x * r1 + y * r2), o2 * (x * r1 - y * r2)


def show(sigma1=1.0, sigma2=1.0, rho=0.0, n=1000):
    x, y = gaussian_samples(sigma1, sigma2, rho, n)
    inside = np.mean((x / sigma1) ** 2 - 2 * rho * x * y / (sigma1 * sigma2) + (y / sigma2) ** 2
                     <= 9 * (1 - rho ** 2))
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.plot(x, y, 'ko', zorder=1, alpha=0.5, ms=2)
    ax.add_artist(contour_ellipse(sigma1, sigma2, rho))
    ax.set_xlim(-6, 6)
    ax.set_ylim(-5, 5)
    ax.set_aspect('equal')
    ax.grid(True)
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')
    ax.set_title(f'n = {n},  Anteil innerhalb der Ellipse: {inside:.3f}  (Theorie: {1 - np.exp(-4.5):.3f})')
    plt.show()

In [ ]:
slider = dict(continuous_update=False, style={'description_width': 'initial'},
              layout=widgets.Layout(width='60%'))
n = widgets.IntSlider(min=100, max=10000, step=100, value=1000, description='Punkte n', **slider)
play = widgets.Play(min=100, max=10000, step=100, interval=150, value=1000)
widgets.jslink((play, 'value'), (n, 'value'))

ui = widgets.interactive(
    show,
    sigma1=widgets.FloatSlider(min=0.1, max=2.0, step=0.01, value=1.0, description='sigma1', **slider),
    sigma2=widgets.FloatSlider(min=0.1, max=2.0, step=0.01, value=1.0, description='sigma2', **slider),
    rho=widgets.FloatSlider(min=-0.99, max=0.99, step=0.01, value=0.0, description='rho', **slider),
    n=n,
)
display(widgets.VBox([play, ui]))